# aeon-qc exploration

Interactive cells for testing QC metrics against benchmark datasets.
See `BENCHMARKS.md` for dataset paths and time ranges.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aeon_qc import heartbeat_gaps, dropped_frames, epoch_gaps, run_qc, save_results, generate_report, schema_from_metadata

## Configuration

Edit these values to point at the dataset you want to inspect.

In [ ]:
ROOT = r"Z:\aeon\data\raw\AEON4\social0.4"
SCHEMA = schema_from_metadata(ROOT)  # reads workflow tag from Metadata.yml â†’ full social04 schema

START = pd.Timestamp("2024-09-13T12:00:00", tz="UTC")
END   = pd.Timestamp("2024-09-13T13:00:00", tz="UTC")

print("Schema devices:")
for name, streams in SCHEMA.items():
    if isinstance(streams, dict):
        print(f"  {name}: {list(streams.keys())}")
    else:
        print(f"  {name}: {type(streams).__name__}")

In [ ]:
from swc.aeon.io.api import load
from swc.aeon.io.reader import Metadata as MetadataReader

# Devices listed in Metadata.yml for this dataset
meta_df = load(ROOT, MetadataReader())
meta_devices = set(meta_df.iloc[0].metadata.Devices.keys())

# Top-level device names in SCHEMA (excluding the Metadata reader entry)
schema_devices = {name for name in SCHEMA.keys() if name != "Metadata"}

only_in_meta = meta_devices - schema_devices
only_in_schema = schema_devices - meta_devices

print(f"Metadata.yml devices : {len(meta_devices)}")
print(f"Schema devices       : {len(schema_devices)}")
print(f"Matched              : {len(meta_devices & schema_devices)}")

if only_in_meta:
    print(f"\nIn Metadata only (not covered by schema): {sorted(only_in_meta)}")
if only_in_schema:
    print(f"\nIn schema only (not in Metadata.yml):     {sorted(only_in_schema)}")
if not only_in_meta and not only_in_schema:
    print("\nAll devices match.")

## Epoch gaps

Detects gaps between consecutive Bonsai session starts within the `START`/`END` window.
`load()` assembles `Metadata.yml` timestamps across all epoch directories in that range,
so a short gap suggests a crash/restart and a long gap a planned stop â€” interpret in context
of your experiment schedule.

In [ ]:
epochs = epoch_gaps(ROOT, start=START, end=END)
n_gaps = int(epochs["gap_duration"].notna().sum())
print(f"{len(epochs)} epoch(s) in range, {n_gaps} inter-epoch gap(s)")
if not epochs.empty:
    if n_gaps > 0:
        real_gaps = epochs["gap_duration"].dropna()
        print(f"Shortest: {real_gaps.min()}, Longest: {real_gaps.max()}")
    display(epochs)

## Heartbeat gaps

Detects periods where a Harp device stops sending heartbeat events (~1 Hz).
Any inter-event gap exceeding the threshold (default: 1.1 seconds) is flagged.
Returns a DataFrame indexed by gap-start time with `duration` and `device` columns.

In [ ]:
from swc.aeon.io.reader import Heartbeat as HeartbeatReader
from aeon_qc.report import iter_readers

hb_devices = {name: reader for name, reader in iter_readers(SCHEMA)
              if isinstance(reader, HeartbeatReader)}

if not hb_devices:
    print("No Heartbeat devices found in this dataset")
else:
    print(f"Found {len(hb_devices)} heartbeat device(s): {list(hb_devices.keys())}\n")
    for device_name, hb_reader in hb_devices.items():
        gaps = heartbeat_gaps(ROOT, hb_reader, start=START, end=END)
        data_ok = gaps.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA on disk")
        elif gaps.empty:
            print(f"{device_name}: 0 gap(s)")
        else:
            print(f"{device_name}: {len(gaps)} gap(s)")
            display(gaps)

## Sync delta

The ClockSynchronizer broadcasts a clock signal (heartbeat) once per second to all Harp
devices, keeping them synchronised. All devices share a common logical counter (`second`);
aligning on that counter lets us compare Harp timestamps across devices and detect drift.

The ClockSynchronizer is used as the reference. Returns one row per second per
non-reference device with columns `second`, `device`, and `delta_seconds`
(positive = device timestamp is ahead of the reference).

In [ ]:
from aeon_qc import sync_delta
from aeon_qc.sync import MIN_DEVICES

if len(hb_devices) < MIN_DEVICES:
    print(f"sync_delta: need at least {MIN_DEVICES} heartbeat devices (found {len(hb_devices)})")
else:
    delta = sync_delta(ROOT, hb_devices, start=START, end=END)
    if delta.empty:
        print("sync_delta: no data")
    else:
        summary = (
            delta.groupby("device")["delta_seconds"]
            .agg(max_abs=lambda s: s.abs().max(), mean_abs=lambda s: s.abs().mean())
            .sort_values("max_abs", ascending=False)
        )
        print("Sync delta per device (seconds):")
        print(summary.to_string())

## Dropped frames

Detects dropped video frames via `hw_counter` jumps.
Returns a DataFrame indexed by the last received frame timestamp with `duration`,
`n_dropped`, `hw_counter_before`, `hw_counter_after`, and `device` columns.

In [ ]:
from swc.aeon.io.reader import Video as VideoReader

camera_devices = {name: reader for name, reader in iter_readers(SCHEMA)
                  if isinstance(reader, VideoReader)}

if not camera_devices:
    print("No camera devices found in this dataset")
else:
    print(f"Found {len(camera_devices)} camera(s): {list(camera_devices.keys())}\n")
    for device_name, cam_reader in camera_devices.items():
        drops = dropped_frames(ROOT, cam_reader, start=START, end=END)
        data_ok = drops.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA")
        elif drops.empty:
            print(f"{device_name}: 0 drop event(s)")
        else:
            total = drops["n_dropped"].sum()
            print(f"{device_name}: {len(drops)} drop event(s), {total} total frames dropped")
            display(drops)

## Encoder gaps

Detects dropped samples in the wheel encoder stream (~500 Hz, 2 ms interval).
Only gaps longer than the threshold (default 1 second) are flagged, ignoring sub-second jitter.
Returns a DataFrame indexed by gap-start time with `duration`, `n_missed`, and `device` columns.

In [ ]:
from aeon_qc import encoder_gaps
from aeon_qc.encoder import DEFAULT_THRESHOLD as ENCODER_THRESHOLD
from swc.aeon.io.reader import Encoder as EncoderReader

encoder_devices = {name: reader for name, reader in iter_readers(SCHEMA)
                   if isinstance(reader, EncoderReader)}

if not encoder_devices:
    print("No Encoder devices found in this dataset")
else:
    print(f"Found {len(encoder_devices)} encoder(s): {list(encoder_devices.keys())}")
    print(f"Gap threshold: {ENCODER_THRESHOLD}\n")
    for device_name, enc_reader in encoder_devices.items():
        gaps = encoder_gaps(ROOT, enc_reader, start=START, end=END)
        data_ok = gaps.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA")
        elif gaps.empty:
            print(f"{device_name}: 0 gap event(s)")
        else:
            total_missed = gaps["n_missed"].sum()
            print(f"{device_name}: {len(gaps)} gap event(s), {total_missed} total missed samples")
            display(gaps)

## Pellet delivery failures

Reports hardware-logged pellet delivery failures from the `MissedPellet` and
`RetriedDelivery` streams. Events are pre-classified by the hardware; no
time-window matching is performed.
Returns a DataFrame indexed by event time with `outcome` (`missed` or `retried`)
and `device` columns.

In [ ]:
from aeon_qc import pellet_failures
from aeon_qc.report import iter_readers

feeder_devices = {}
for name, reader in iter_readers(SCHEMA):
    device, _, stream = name.partition(".")
    if stream in ("DeliverPellet", "MissedPellet", "RetriedDelivery"):
        feeder_devices.setdefault(device, {})[stream] = reader

feeder_devices = {d: s for d, s in feeder_devices.items() if "DeliverPellet" in s}

if not feeder_devices:
    print("No feeder devices found in schema")
else:
    for device_name, streams in feeder_devices.items():
        stats = pellet_failures(
            ROOT,
            deliver_reader=streams["DeliverPellet"],
            missed_reader=streams.get("MissedPellet"),
            retried_reader=streams.get("RetriedDelivery"),
            start=START, end=END,
        )
        n = stats.attrs.get("n_deliveries", 0)
        n_retried = stats.attrs.get("n_retried", 0)
        n_missed = stats.attrs.get("n_missed", 0)
        print(f"{device_name}: {n} deliveries, {n_retried} retried, {n_missed} missed")
        if not stats.empty:
            display(stats)

## Message log errors

Surfaces Warning and Error entries from the Bonsai message log.
Info-level entries are excluded. Useful for detecting software crashes or
unexpected hardware states during data collection.

In [ ]:
from aeon_qc import message_log_errors
from swc.aeon.io.reader import Log as LogReader

log_readers = {name: reader for name, reader in iter_readers(SCHEMA)
               if isinstance(reader, LogReader)}

if not log_readers:
    print("No MessageLog readers found in schema")
else:
    for name, reader in log_readers.items():
        errors = message_log_errors(ROOT, reader, start=START, end=END)
        n_total = errors.attrs.get("n_total", 0)
        print(f"{name}: {n_total} total log entries, {len(errors)} non-Info")
        if not errors.empty:
            display(errors)

## Environment state

Shows time spent in each environment state (e.g. `Running`, `Maintenance`) over
the selected time range.

In [ ]:
from aeon_qc import environment_state_durations
from swc.aeon.io.reader import Csv as CsvReader

env_state_readers = {name: reader for name, reader in iter_readers(SCHEMA)
                     if isinstance(reader, CsvReader) and "EnvironmentState" in name}

if not env_state_readers:
    print("No EnvironmentState readers found in schema")
else:
    for name, reader in env_state_readers.items():
        durations = environment_state_durations(ROOT, reader, start=START, end=END)
        if not durations.attrs.get("data_found", True):
            print(f"{name}: NO DATA")
        elif durations.empty:
            print(f"{name}: no state transitions found")
        else:
            totals = durations.groupby("state")["duration"].sum()
            print(f"{name}:")
            for state, total in totals.items():
                print(f"  {state}: {total}")

## Full scan

Runs all applicable QC checks across every stream in the schema at once.
Returns a dict of DataFrames keyed by device stream name.

In [ ]:
results = run_qc(ROOT, SCHEMA, start=START, end=END)

for name, df in results.items():
    found = df.attrs.get("data_found", True)
    status = "NO DATA" if not found else f"{len(df)} row(s)"
    print(f"{name}: {status}")

## Generate YAML report

Writes a structured YAML summary to disk with per-device `summary` and `detail` sections.
Also saves the full results dict as a pickle for downstream analysis.

In [ ]:
import os

os.makedirs("qc_results", exist_ok=True)

output = generate_report(ROOT, results, "qc_results/qc_report.yaml", start=START, end=END)
print(f"Report written to {output}")

pkl = save_results(results, "qc_results/qc_results.pkl")
print(f"Results pickled to {pkl}")